# KP3200 Python Lab Skeleton Code 2026

This is the skeleton code you will need to complete the tasks in the Python Lab. I recommend using [NTNUs Jupyter Hub Server](https://jupyter.ntnu.no/) to run Python code. The advantage of running code on this server is that all the packages we need has already been installed, which means we do not need to install anything! If you wish to read more about NTNU's Jupyter Hub server, [click here](https://www.ntnu.no/wiki/spaces/jupyterhub/pages/178652271/JupyterHub+at+NTNU).

If you run code on your local machine, you may want to install necessary packages to a virtual environment using pip -- I assume those of you that prefer to run code locally also knows how to do it, but feel free to reach out to us in the exercise sessions if you need help with setting up a Jupyter Notebook locally on your machine or want to learn more about this. Aksel can help.

# Part 1: Mass Balances

In [ ]:
# First, we need to import some packages:
import numpy as np                # For some mathematical operations
from scipy.optimize import fsolve # For the numerical solver

# I recommend following the same notation scheme as the skeleton code uses, i.e., using the same variable names.
# If you change the naming scheme, make sure to update all variables.

# Let's start with writing down the given info from the Lab Manual.
#https://studntnu-my.sharepoint.com/:w:/g/personal/andrglos_ntnu_no/IQDVb_L9A9gpTYPv7JgqADCoAX3if2EQXEMrxSCG-fm9AZI?e=Z0QEJ0
# Given info
n_1 =                  # mol/s 
n_2 =                  # mol/s
n_5 =                  # mol/s

x_1O2 =                # O2 mole fraction in stream 1
x_6NH3 =               # NH3 mole fraction in stream 3
x_3N2 =                # Nitrogen mole fraction in stream 6
x_4NH3 =               # NH3 mole fraction in stream 7

# Directly inferred info
x_1N2 =                # N2 mole fraction in stream 1

def eqns(u):
    x_6O2, x_6N2, x_3O2, x_3NH3, x_4O2, x_4N2, n_6, n_7, n_3, n_4 = u # List of the 10 unknown variables in this specific order. 
    
    f = np.empty(len(u)) # Defining an empty vector the same size as u, which will later be used to contain the unknowns

    # We now begin writing the mole balance equations on the form f(x) = 0. # Example: n_1 + n_2 = n_6 becomes f[0] = n_1 + n_2 - n_6. These functions defined at 0 are called residuals.
    
    f[0] =                            # Total mole balance overall
    f[1] =                            # NH3 balance overall
    f[2] =                            # O2 balance overall
    f[3] =                            # Total mole balance over mixer
    f[4] =                            # NH3 balance over mixer
    f[5] =                            # O2 balance over mixer
    f[6] =                            # Total mole balance over split-point
    f[7] =                            # Sum of all mole fractions in stream 3
    f[8] =                            # Sum of all mole fractions in stream 6
    f[9] =                            # Sum of all mole fractions in stream 7
    
    return f # Returns our equations as a matrix.

guess = np.array([   ]) # This is our initial guess for the numerical solver: the exact numbers can vary, as long as they are physically possible (i.e. no non-zero numbers and fraction guesses for molar fractions)
sol = fsolve(eqns, guess)
x_6O2, x_6N2, x_3O2, x_3NH3, x_4O2, x_4N2, n_6, n_7, n_3, n_4 = sol

# Now, let's print out the unknowns:

np.set_printoptions(suppress=True, precision=5) # Print setting for arrays to make them more readable

# You may use f-strings: print(f"Regular text {variable}")

# Part 2: Energy Balance

In [ ]:
# We can now finally calculate the energy requirement (and ultimately the heating flow rate) for the exiting stream 6.

# Begin with defining known variables:

T_in =                          # Ingoing temperature (°C)
T_out =                         # The target final temperature (K) of the gas
T_cold =                        # Temperature of outgoing heating stream in °C
T_warm =                        # Temperature of ingoing heating stream in °C
M_H2O =                         # kg/mol

# Now, finding the mean (average) isobaric heat capacity for NH3, O2 and N2:

def heatcapacity_gas(T_in, T_out): # Mean heat capacity for gas from T_in to T_out [kJ/(mol*°C)] (Felder et al. 2016, Table B.2)
    Heatcoeff_gas_NH3 = [35.15e-03, 2.954e-05, 0.4421e-08, -6.686e-12]  # Heat capacity coefficients (a,b,c,d) for gas phase, all valid in the 0-1000 °C range
    Heatcoeff_gas_O2 = [29.10e-03, 1.158e-05, -0.6076e-08, 1.311e-12]
    Heatcoeff_gas_N2 = [29.00e-03, 0.2199e-05, 0.5723e-08, -2.871e-12]

    # Creates a coefficient matrix
    coeff_gas = np.array([Heatcoeff_gas_NH3, Heatcoeff_gas_O2, Heatcoeff_gas_N2]).T # Transposes the matrix

    # Calculates Cp_gas_i for each component
    Cp_gas_i = (coeff_gas[0, :] * (T_out - T_in)
                    + 1/2 * coeff_gas[1, :] * (T_out**2 - T_in**2) 
                    + 1/3 * coeff_gas[2, :] * (T_out**3 - T_in**3) 
                    + 1/4 * coeff_gas[3, :] * (T_out**4 - T_in**4)) / (T_out - T_in)
    return Cp_gas_i

# Isobaric heat capacity of N2 gas (heating medium)

def heatcapacity_heating_medium(T_warm, T_cold): # Mean heat capacity for gas from T_in to T_out [kJ/(mol*°C)] (Felder et al. 2016, Table B.2)
    Heatcoeff_gas_N2 = [29.00e-03, 0.2199e-05, 0.5723e-08, -2.871e-12]  # Heat capacity coefficients (a,b,c,d) for gas phase, all valid in the 0-1000 °C range

    # Creates a coefficient matrix
    coeff_gas = np.array([Heatcoeff_gas_N2]).T # Transposes the matrix

    # Calculates Cp_gas_i for each component
    Cp_gas_i = (coeff_gas[0] * (T_warm - T_cold)
                    + 1/2 * coeff_gas[1] * (T_warm**2 - T_cold**2) 
                    + 1/3 * coeff_gas[2] * (T_warm**3 - T_cold**3) 
                    + 1/4 * coeff_gas[3] * (T_warm**4 - T_cold**4)) / (T_warm - T_cold)
    return Cp_gas_i

# Then, write the expressions for the per-species energy requirement using the change in enthalpy for the different species:

Q_NH3 =   # Energy requirement for NH3 in kJ/s
Q_O2 =    # Energy requirement for O2 in kJ/s
Q_N2 =    # Energy requirement for N2 in kJ/s

Q_total = # Total energy requirement in kJ/s

# Having now found the energy requirement for the operation, we can calculate the necessary flow rate of the heating stream (assuming perfect heat transfer/no heat loss)
# Write out the expression for the energy balance (Q_1 = Q_2) using the enthalpy change of the heating medium (N2), then rearrange and solve for the flow rate of the heating stream

n_heating =   # mol/s

m_heating =   # kg/s

In [ ]:
# Tester Tester om github fuhnkerrrrrr